# ChatBS Analysis 2

Run ChatBS evaluations without BERTScore, LLM judge, or NLI metrics; add entity retrieval metrics to every category; and display only the overall summary table. Edit the config cell below instead of passing command line arguments.

In [1]:
from pathlib import Path

# Config parameters. These mirror the CLI arguments in chatbs_analysis2.py.
EVALUATION = "chatbs-base"

# None writes to evaluations/<EVALUATION>/analysis2. If the folder already has
# files, a timestamped subfolder is used unless OVERWRITE is True.
SAVE_DIR = None

# None evaluates all configured methods. Example: ["fullcontext", "ours"]
METHODS = None

# None evaluates every available prediction row.
MAX_EXAMPLES_PER_RUN = None

# None uses metrics.answer_report from config.evaluation.yaml.
ANSWER_REPORT = None

# True displays the table without writing CSV outputs.
NO_SAVE = False

# True allows writing directly into SAVE_DIR even if it already has files.
OVERWRITE = False

ROUND_DIGITS = 3

In [2]:
import importlib.util

import pandas as pd
from IPython.display import display

candidate_scripts = [
    Path("evaluations/biomni-base/biomni_analysis2.py"),
    Path("../biomni-base/biomni_analysis2.py"),
    Path.cwd().parent / "biomni-base" / "biomni_analysis2.py",
]
SCRIPT_PATH = next((path for path in candidate_scripts if path.exists()), None)
if SCRIPT_PATH is None:
    raise FileNotFoundError("Could not find evaluations/biomni-base/biomni_analysis2.py")
SCRIPT_PATH = SCRIPT_PATH.resolve()

spec = importlib.util.spec_from_file_location("analysis2_common", SCRIPT_PATH)
analysis2 = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(analysis2)

analysis2.load_dotenv(analysis2.REPO_ROOT / ".env")
SCRIPT_PATH

PosixPath('/home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/biomni_analysis2.py')

In [3]:
evaluation_dir = analysis2.resolve_repo_path(Path("evaluations") / EVALUATION)
config_path = evaluation_dir / "config.evaluation.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"Evaluation config not found: {config_path}")

evaluation_config = analysis2.load_config(str(config_path))
settings = evaluation_config.get("evaluation", {})
metrics_config = settings.get("metrics", {})
metrics_by_qtype = analysis2.configured_metrics(
    metrics_config.get("enabled_by_qtype", {}),
)

prediction_dirs = dict(settings.get("prediction_dirs", {}))
configured_prediction_files = dict(settings.get("prediction_files", {}))
available_methods = set(prediction_dirs) | set(configured_prediction_files)
selected_methods = set(METHODS) if METHODS is not None else None
if selected_methods is not None:
    unknown_methods = selected_methods - available_methods
    if unknown_methods:
        raise ValueError(f"Unknown methods: {sorted(unknown_methods)}")

prediction_files = analysis2.resolve_prediction_files(
    prediction_dirs,
    configured_prediction_files,
    settings.get("prediction_filename", "RESULTS.jsonl"),
    selected_methods,
)

answer_report = ANSWER_REPORT or metrics_config.get(
    "answer_report",
    settings.get("answer_report", "original"),
)
input_augmentations = analysis2.build_input_augmentation_map(
    selected_methods,
    settings,
    answer_report,
)

gt_bundle = analysis2.load_ground_truth_bundle(
    EVALUATION,
    settings.get("source_config", "config.fullcontext.yaml"),
)

print(f"Config: {config_path}")
print(f"Ground truth: {gt_bundle['ground_truth_path']}")
print(f"Ground-truth examples: {len(gt_bundle['records'])}")
print(f"Enabled qtype metrics: {', '.join(sorted(metrics_by_qtype))}")
print("Removed metrics: bert_score, llm_answer_quality, nli_entailment")
print("Forced metric on every qtype: entity_retrieval")

Config: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/config.evaluation.yaml
Ground truth: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/ground_truth/ground_truth_data.jsonl
Ground-truth examples: 103
Enabled qtype metrics: bool, entity, numeric
Removed metrics: bert_score, llm_answer_quality, nli_entailment
Forced metric on every qtype: entity_retrieval


In [4]:
prediction_runs = analysis2.load_prediction_runs(
    prediction_files,
    input_augmentations,
    MAX_EXAMPLES_PER_RUN,
)

evaluation_rows = {}
for run_name, predictions in prediction_runs.items():
    run_rows = analysis2.evaluate_run(run_name, predictions, metrics_by_qtype, gt_bundle)
    for category, rows in run_rows.items():
        evaluation_rows.setdefault(category, []).extend(rows)

categories_to_write = sorted(set(metrics_by_qtype) | set(evaluation_rows))
results_by_category = {}
summaries_by_category = {}
for category in categories_to_write:
    results_df = analysis2.sort_results_df(pd.DataFrame(evaluation_rows.get(category, [])))
    summary_df = analysis2.build_run_summary(results_df)
    results_by_category[category] = results_df
    summaries_by_category[category] = summary_df
    print(f"{category}: {len(results_df)} evaluated rows")

overall_summary_df = analysis2.build_overall_summary(results_by_category)
overall_table = analysis2.format_overall_table(overall_summary_df, ROUND_DIGITS)

display(overall_table)

ours: 100%|██████████| 103/103 [00:03<00:00, 33.16it/s]

bool: 266 evaluated rows
entity: 343 evaluated rows
numeric: 112 evaluated rows


,Method,N,Matched GT,Answer Token F1,GT Entity Coverage,Entity Recall Final,Entity Precision Final,Entity F1 Final,Entity Recall Total,Entity Precision Total,Entity F1 Total
0,FCB,103,103,36.616,23.354,50.348,15.167,35.122,55.570,10.674,25.846
1,GWB,103,103,18.563,0.000,23.364,4.169,17.508,35.512,6.362,19.229
2,VSB,103,103,23.582,0.973,9.609,2.691,17.542,14.488,2.327,13.687
3,GRASP,103,103,8.633,0.165,0.000,NaN,NaN,0.000,NaN,NaN
4,HippoRAG,103,103,6.530,10.447,25.799,1.015,5.102,57.258,1.171,3.737
5,HyperGRAG,103,103,22.946,5.062,0.000,NaN,NaN,0.000,NaN,NaN
6,Ours,103,103,30.244,19.818,50.610,8.683,15.058,62.122,2.870,7.170


In [5]:
output_dir = None
if not NO_SAVE:
    default_save_dir = evaluation_dir / "analysis2"
    requested_save_dir = analysis2.resolve_repo_path(SAVE_DIR) if SAVE_DIR else default_save_dir
    output_dir = analysis2.prepare_output_dir(requested_save_dir, overwrite=OVERWRITE)
    analysis2.write_outputs(output_dir, results_by_category, summaries_by_category, overall_table)
    print(f"Wrote outputs under: {output_dir}")
else:
    print("NO_SAVE is True, so no files were written.")

Wrote outputs under: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/analysis2/run_20260507_163437
